# Serial Killer Typology Discovery (v2, corrected analysis)

**What changed from v1:** features are log-transformed and standardized (v1 left `decade` unscaled, so it dominated PCA and K-Means); clustering uses *behavioural* features only (era and region are used afterwards to describe clusters); country one-hot (71 columns) is replaced by 9 regions; the severity classifier no longer leaks victim counts; the ARIMA series no longer imputes missing years to 1981; and every model is validated with cross-validation or bootstrap, not on the data it was fitted on.

Dataset: Kaggle `abdullahsamiir/serial-killers-wiki` (Wikipedia-derived). Put `serial_killers.csv` in `data/` or let the notebook download it with `kagglehub`.

In [ ]:
import os, re, json, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
warnings.filterwarnings("ignore")

SEED = 42
np.random.seed(SEED)
FIG_DIR, RES_DIR = "figures", "results"
os.makedirs(FIG_DIR, exist_ok=True); os.makedirs(RES_DIR, exist_ok=True)
sns.set_theme(style="whitegrid")

def savefig(name):
    plt.tight_layout()
    plt.savefig(f"{FIG_DIR}/{name}.png", dpi=150, bbox_inches="tight")

SUMMARY = {}   # key numbers collected here and saved at the end

In [ ]:
def load_data():
    for p in [os.environ.get("SK_DATA", ""), "data/serial_killers.csv", "../data/serial_killers.csv"]:
        if p and os.path.exists(p):
            return pd.read_csv(p)
    import kagglehub  # needs internet + (for some setups) Kaggle credentials
    path = kagglehub.dataset_download("abdullahsamiir/serial-killers-wiki")
    return pd.read_csv(os.path.join(path, "serial_killers.csv"))

raw = load_data()
raw.columns = [c.strip() for c in raw.columns]
print(raw.shape)
raw.head()

## 1. Cleaning and feature engineering

Fixes vs v1: citation markers are stripped *before* parsing; `1990s` now means 1990–1999 (v1 gave 1990–2089); `Country` is split on `, ; /` only (v1 split on spaces, which turned "United States" into "United"); "numerous" is no longer silently converted to 1000; nothing is dropped because of the `Name` column.

In [ ]:
YEAR_RE = re.compile(r"(?<!\d)(1[5-9]\d{2}|20\d{2})(?!\d)")
CURRENT_YEAR = 2026

def parse_years_active(s):
    if pd.isna(s):
        return np.nan, np.nan
    s = str(s).replace("\xa0", " ").replace("\u2013", "-").replace("\u2014", "-")
    s = re.sub(r"\[[^\]]*\]", "", s).strip()
    m = re.fullmatch(r"(1[5-9]\d0|20\d0)s", s)           # e.g. "1990s"
    if m:
        y = int(m.group(1)); return y, y + 9
    years = [int(y) for y in YEAR_RE.findall(s)]
    if not years:
        return np.nan, np.nan
    return min(years), min(max(years), CURRENT_YEAR)

def clean_victim_counts(v):
    if pd.isna(v):
        return np.nan
    t = re.sub(r"\[[^\]]*\]", "", str(v)).strip().lower().replace(",", "")
    if t in {"", "unconfirmed", "unknown", "na", "n/a"} or "numerous" in t:
        return np.nan                                   # unquantified -> missing, not an invented number
    if "hundreds" in t:
        return 100.0                                    # lower bound, consistent with range handling
    nums = re.findall(r"\d+", t)
    return float(nums[0]) if nums else np.nan           # ranges / "300+" -> lower bound

df = raw.copy()
yrs = df["Years active"].apply(parse_years_active)
df["start_year"] = yrs.apply(lambda t: t[0]); df["end_year"] = yrs.apply(lambda t: t[1])
df["duration"] = df["end_year"] - df["start_year"] + 1
df["proven"] = df["Proven victims"].apply(clean_victim_counts)
df["possible"] = df["Possible victims"].apply(clean_victim_counts)

print("unparsable years:", df["start_year"].isna().sum(),
      "| missing proven:", df["proven"].isna().sum(), "| missing possible:", df["possible"].isna().sum())
print("\nLongest durations (check for parse errors):")
display(df.nlargest(6, "duration")[["Name", "Years active", "start_year", "end_year", "duration"]])

In [ ]:
REGIONS = {
 "North America": ["united states", "canada"],
 "Latin America": ["mexico","brazil","colombia","argentina","peru","chile","ecuador","venezuela","cuba","guatemala",
                   "honduras","el salvador","costa rica","panama","bolivia","uruguay","paraguay","dominican",
                   "puerto rico","jamaica","nicaragua"],
 "Western Europe": ["united kingdom","england","scotland","wales","ireland","france","germany","italy","spain",
                    "portugal","netherlands","belgium","switzerland","austria","sweden","norway","denmark",
                    "finland","iceland","greece"],
 "Eastern Europe & Russia": ["russia","soviet","ukraine","poland","czech","hungary","romania","bulgaria","serbia",
                             "yugoslavia","croatia","slovakia","belarus","lithuania","latvia","estonia","moldova",
                             "bosnia","slovenia"],
 "Middle East": ["iran","iraq","turkey","israel","saudi","syria","lebanon","jordan","afghanistan"],
 "Africa": ["south africa","nigeria","kenya","egypt","uganda","ghana","zimbabwe","tanzania","algeria","morocco",
            "ethiopia","zambia","namibia","congo","cameroon"],
 "Asia": ["china","japan","india","korea","thailand","vietnam","philippines","indonesia","malaysia","pakistan",
          "bangladesh","taiwan","sri lanka","nepal","kazakhstan","uzbekistan","singapore","cambodia","myanmar"],
 "Oceania": ["australia","new zealand","papua"],
}
def primary_country(c):
    if pd.isna(c): return "Unknown"
    c = re.sub(r"\([^)]*\)|\[[^\]]*\]", "", str(c))
    return re.split(r"[,;/]", c)[0].strip() or "Unknown"

def to_region(c):
    c = c.lower()
    for region, keys in REGIONS.items():
        if any(k in c for k in keys):
            return region
    return "Other"

df["country"] = df["Country"].apply(primary_country)
df["region"] = df["country"].apply(to_region)
print(df["region"].value_counts())
print("\nCountries mapped to 'Other' (extend REGIONS if any matter):")
print(df.loc[df["region"] == "Other", "country"].value_counts().head(15))

### De-duplication
The source CSV merges lists that name the same offender in different formats ("Aileen Wuornos" and "Wuornos, Aileen") and sometimes repeats rows verbatim. Duplicates inflate counts (including the 1986 peak) and bias clusters, so rows with the same normalised name and start year are collapsed, keeping the row with the most victim information. Name variants that differ beyond word order (e.g. a pair listed jointly vs. individually) are not caught, so a few duplicates may remain.

In [ ]:
import unicodedata

def name_key(n):
    n = unicodedata.normalize("NFKD", str(n))
    n = "".join(ch for ch in n if not unicodedata.combining(ch))
    n = re.sub(r"\[[^\]]*\]|\([^)]*\)", "", n).strip().lower()
    if "," in n:                                   # "Last, First" -> "First Last"
        last, first = n.split(",", 1); n = first.strip() + " " + last.strip()
    n = re.sub(r"[^a-z0-9 ]", "", n)
    return " ".join(sorted(n.split()))             # ignore word order

df["name_key"] = df["Name"].apply(name_key)
df["_info"] = df["proven"].notna().astype(int) + df["possible"].notna().astype(int)
n_before = len(df)
df = (df.sort_values("_info", ascending=False, kind="stable")
        .drop_duplicates(["name_key", "start_year"]).sort_index().drop(columns="_info"))
print(f"Removed {n_before - len(df)} duplicate rows ({n_before} -> {len(df)})")
SUMMARY["n_duplicates_removed"] = int(n_before - len(df))

In [ ]:
# Modelling sample: need a start year and at least one victim count. No row is dropped for Name.
m = df.dropna(subset=["start_year"]).copy()
m = m[~(m["proven"].isna() & m["possible"].isna())].copy()
m["victims_imputed"] = m["proven"].isna() | m["possible"].isna()
m["proven"] = m["proven"].fillna(m["possible"])          # if only one is known, use it for both
m["possible"] = m["possible"].fillna(m["proven"])
m["possible"] = np.maximum(m["possible"], m["proven"])   # possible >= proven by definition
m["decade"] = (m["start_year"] // 10 * 10).astype(int)

n_clipped = (m["duration"] > 60).sum()
m["duration"] = m["duration"].clip(1, 60)                # winsorise absurd spans
m["log_proven"] = np.log1p(m["proven"])
m["log_possible"] = np.log1p(m["possible"])
m["log_duration"] = np.log1p(m["duration"])

print(f"Rows: {len(raw)} raw -> {len(m)} modelling sample ({len(raw)-len(m)} dropped; v1 dropped 273)")
print(f"Victim counts partly imputed in {m['victims_imputed'].sum()} rows | durations clipped at 60y: {n_clipped}")
SUMMARY.update(n_raw=int(len(raw)), n_model=int(len(m)))

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(15, 4))
for a, (col, t) in zip(ax, [("proven", "Proven victims"), ("possible", "Possible victims"), ("duration", "Active duration (years)")]):
    sns.histplot(m[col], bins=40, log_scale=(False, True), ax=a); a.set_title(t + " (log y-axis)")
savefig("01_distributions")
plt.show()

## 2. Clustering on behavioural features only

Features: log proven victims, log possible victims, log active duration, standardized. **Era (decade) and region are deliberately excluded** so the clusters cannot be a time-period artefact; they are used afterwards to describe the clusters.

In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans, AgglomerativeClustering, DBSCAN
from sklearn.mixture import GaussianMixture
from sklearn.decomposition import PCA
from sklearn.metrics import (silhouette_score, davies_bouldin_score, calinski_harabasz_score,
                             adjusted_rand_score, adjusted_mutual_info_score)

BEHAV = ["log_proven", "log_possible", "log_duration"]
X = StandardScaler().fit_transform(m[BEHAV])

rows = []
for k in range(2, 9):
    lab = KMeans(k, n_init=20, random_state=SEED).fit(X)
    rows.append(dict(k=k, inertia=lab.inertia_, silhouette=silhouette_score(X, lab.labels_),
                     davies_bouldin=davies_bouldin_score(X, lab.labels_),
                     calinski_harabasz=calinski_harabasz_score(X, lab.labels_),
                     smallest_cluster=int(np.bincount(lab.labels_).min())))
ksel = pd.DataFrame(rows)
display(ksel.round(3))

# choose K in 3..6 with best silhouette, requiring every cluster to hold >=5% of the data
cand = ksel[ksel.k.between(3, 6) & (ksel.smallest_cluster >= 0.05 * len(m))]
K = int(cand.sort_values("silhouette").iloc[-1]["k"]) if len(cand) else 3
print("Chosen K =", K)

fig, ax = plt.subplots(1, 4, figsize=(17, 3.6))
for a, c, t in zip(ax, ["inertia", "silhouette", "davies_bouldin", "calinski_harabasz"],
                   ["Inertia (elbow)", "Silhouette (higher better)", "Davies-Bouldin (lower better)", "Calinski-Harabasz (higher better)"]):
    a.plot(ksel.k, ksel[c], marker="o"); a.axvline(K, color="red", ls="--", alpha=.5); a.set_title(t); a.set_xlabel("K")
savefig("02_k_selection")
plt.show()
SUMMARY["K"] = K

In [ ]:
models = {
    "KMeans": KMeans(K, n_init=20, random_state=SEED),
    "GaussianMixture": GaussianMixture(K, n_init=5, random_state=SEED),
    "Agglomerative (ward)": AgglomerativeClustering(K),
    "DBSCAN (eps=0.6, min=10)": DBSCAN(eps=0.6, min_samples=10),
}
labels = {n: mdl.fit_predict(X) for n, mdl in models.items()}
rows = []
for n, l in labels.items():
    core = l != -1
    ncl = len(set(l[core]))
    sil = silhouette_score(X[core], l[core]) if ncl >= 2 and core.sum() > ncl else np.nan
    rows.append(dict(method=n, clusters=ncl, noise_points=int((~core).sum()), silhouette=sil,
                     ARI_vs_KMeans=adjusted_rand_score(labels["KMeans"], l)))
comparison = pd.DataFrame(rows)
display(comparison.round(3))
comparison.to_csv(f"{RES_DIR}/clustering_method_comparison.csv", index=False)

In [ ]:
# Stability: refit K-Means on 50 bootstrap resamples and compare to the reference partition
ref = KMeans(K, n_init=20, random_state=SEED).fit(X)
rng = np.random.RandomState(SEED)
aris = []
for b in range(50):
    idx = rng.choice(len(X), len(X), replace=True)
    kb = KMeans(K, n_init=10, random_state=b).fit(X[idx])
    aris.append(adjusted_rand_score(ref.labels_, kb.predict(X)))
print(f"Bootstrap stability (ARI vs reference): mean={np.mean(aris):.3f}, min={np.min(aris):.3f}")
SUMMARY["stability_ari_mean"] = float(np.mean(aris))

# Name clusters in order of proven-victim level so labels are stable and readable
order = pd.Series(ref.labels_).groupby(ref.labels_).apply(lambda s: m["log_proven"].values[s.index].mean()).sort_values().index
remap = {old: new for new, old in enumerate(order)}
m["cluster"] = [remap[l] for l in ref.labels_]
SUMMARY["silhouette"] = float(silhouette_score(X, m["cluster"]))
print("Silhouette:", round(SUMMARY["silhouette"], 3)); print(m["cluster"].value_counts().sort_index())

In [ ]:
pca = PCA(2, random_state=SEED).fit(X)
P = pca.transform(X)
plt.figure(figsize=(8, 6))
sns.scatterplot(x=P[:, 0], y=P[:, 1], hue=m["cluster"].values, palette="viridis", s=25, alpha=.7)
plt.xlabel(f"PC1 ({pca.explained_variance_ratio_[0]*100:.1f}% var)"); plt.ylabel(f"PC2 ({pca.explained_variance_ratio_[1]*100:.1f}% var)")
plt.title("Behavioural clusters in PCA space")
savefig("03_clusters_pca"); plt.show()
print("Variance explained by PC1, PC2:", pca.explained_variance_ratio_.round(3))

## 3. Describing the clusters (era and region *after* clustering)

If the clusters were just time periods (as in v1), `AMI(cluster, decade)` would be near 1. Kruskal-Wallis is used as a **descriptive** check only: the clusters were built from these same variables, so p-values here are not independent validation.

In [ ]:
from scipy.stats import kruskal
prof = m.groupby("cluster").agg(n=("cluster", "size"), median_proven=("proven", "median"), median_possible=("possible", "median"),
                                median_duration=("duration", "median"), median_start_year=("start_year", "median"))
display(prof)
print("Region share per cluster (%):")
display((pd.crosstab(m["cluster"], m["region"], normalize="index") * 100).round(1))

era = pd.cut(m["decade"], [0, 1900, 1950, 1970, 1990, 2000, 2010, 3000], right=False,
             labels=["pre-1900", "1900-49", "1950-69", "1970-89", "1990s", "2000s", "2010s+"])
print("Era share per cluster (%):")
display((pd.crosstab(m["cluster"], era, normalize="index") * 100).round(1))

ami_era = adjusted_mutual_info_score(m["cluster"], m["decade"])
ami_beh = adjusted_mutual_info_score(m["cluster"], pd.qcut(m["log_proven"], 5, duplicates="drop", labels=False))
print(f"AMI(cluster, decade) = {ami_era:.3f}  (v1 clusters were ~time periods)")
SUMMARY["ami_cluster_decade"] = float(ami_era)

for col in ["proven", "possible", "duration", "start_year"]:
    H, p = kruskal(*[g[col].values for _, g in m.groupby("cluster")])
    print(f"Kruskal-Wallis {col:11s} H={H:8.2f}  p={p:.2e}  (descriptive)")
prof.to_csv(f"{RES_DIR}/cluster_profile.csv")

In [ ]:
# Surrogate tree: used ONLY to explain the clusters in plain rules (it predicts labels from the same features, so its accuracy is not evidence).
from sklearn.tree import DecisionTreeClassifier, export_text
from sklearn.model_selection import cross_val_score
tree = DecisionTreeClassifier(max_depth=3, random_state=SEED)
print("Surrogate tree 5-fold CV accuracy:", cross_val_score(tree, m[BEHAV], m["cluster"], cv=5).mean().round(3))
tree.fit(m[BEHAV], m["cluster"])
print(export_text(tree, feature_names=BEHAV))

## 4. Using the `Notes` column: themes and text clusters

Keyword flags give interpretable behaviour features. TF-IDF + SVD + K-Means then finds text-based groups without any hand-written keywords. (For higher quality, swap in `sentence-transformers` embeddings; it needs a model download so it is not required here.)

In [ ]:
THEMES = {
 "poison": r"poison|arsenic|cyanide|strychnine|thallium",
 "sexual_violence": r"\brape|sexual|sex ",
 "financial_motive": r"robber|insurance|money|financial|inherit|burglar",
 "healthcare": r"nurse|doctor|physician|hospital|patient|medical|euthan|caregiver",
 "children": r"child|infant|baby|\bboys?\b|\bgirls?\b|teen",
 "sex_workers": r"prostitut|sex worker|streetwalk",
 "torture_ritual": r"torture|ritual|cult|satan|sacrific",
 "mutilation": r"dismember|necrophil|cannibal|mutilat",
 "strangulation": r"strangl|asphyx|suffocat",
 "firearm": r"\bshot\b|shooting|firearm|\bgun",
 "accomplices": r"couple|accomplice|husband and wife|duo|brothers|gang",
 "military": r"soldier|military|\bwar\b|\barmy\b",
 "contract_killing": r"hitman|contract kill|mafia|organi[sz]ed crime",
}
notes = m["Notes"].fillna("").str.lower()
for t, rx in THEMES.items():
    m["theme_" + t] = notes.str.contains(rx, regex=True).astype(int)
THEME_COLS = ["theme_" + t for t in THEMES]
print("Theme prevalence (%):"); print((m[THEME_COLS].mean() * 100).round(1).sort_values(ascending=False))
print("\nTheme prevalence per behavioural cluster (%):")
display((m.groupby("cluster")[THEME_COLS].mean() * 100).round(1).T)

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import TruncatedSVD
from sklearn.preprocessing import Normalizer

tfidf = TfidfVectorizer(stop_words="english", min_df=5, max_df=0.5, ngram_range=(1, 2), max_features=5000, sublinear_tf=True)
T = tfidf.fit_transform(m["Notes"].fillna(""))
Z = Normalizer().fit_transform(TruncatedSVD(min(50, T.shape[1] - 1), random_state=SEED).fit_transform(T))

best = max(((k, silhouette_score(Z, KMeans(k, n_init=10, random_state=SEED).fit_predict(Z))) for k in range(4, 9)), key=lambda t: t[1])
KT = best[0]; print(f"Text clusters: K={KT}, silhouette={best[1]:.3f}")
m["text_cluster"] = KMeans(KT, n_init=20, random_state=SEED).fit_predict(Z)
terms = np.array(tfidf.get_feature_names_out())
for c in range(KT):
    idx = np.where(m["text_cluster"].values == c)[0]
    top = terms[np.argsort(np.asarray(T[idx].mean(axis=0)).ravel())[::-1][:10]]
    print(f"Text cluster {c} (n={len(idx)}): {', '.join(top)}")
SUMMARY["text_K"] = int(KT)

## 5. Victim-severity classifier (no leakage)

v1 predicted a severity class derived from **proven victims** using **possible victims** as a feature, which is nearly the same number. Here victim counts are excluded entirely; features are duration, decade, region and Notes themes. A "leaky" model is shown for comparison. All scores are 5-fold stratified cross-validation; macro-F1 is reported because classes are imbalanced.

In [ ]:
from sklearn.model_selection import StratifiedKFold, cross_validate, cross_val_predict
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.dummy import DummyClassifier
from sklearn.metrics import ConfusionMatrixDisplay

m["severity"] = pd.cut(m["proven"], [-1, 5, 15, 50, np.inf], labels=["Low", "Medium", "High", "Extreme"])
print(m["severity"].value_counts().sort_index())

region_d = pd.get_dummies(m["region"], prefix="region", dtype=int)
Xs = pd.concat([m[["log_duration", "decade"]], region_d, m[THEME_COLS]], axis=1)
Xs_leaky = Xs.assign(log_possible=m["log_possible"])
y = m["severity"].astype(str)

cv = StratifiedKFold(5, shuffle=True, random_state=SEED)
cands = {
    "Majority-class baseline": (DummyClassifier(strategy="most_frequent"), Xs),
    "Decision tree (depth 5)": (DecisionTreeClassifier(max_depth=5, min_samples_leaf=10, class_weight="balanced", random_state=SEED), Xs),
    "Random forest": (RandomForestClassifier(300, min_samples_leaf=3, class_weight="balanced_subsample", random_state=SEED, n_jobs=-1), Xs),
    "Gradient boosting": (GradientBoostingClassifier(random_state=SEED), Xs),
    "Random forest + possible victims (LEAKY)": (RandomForestClassifier(300, min_samples_leaf=3, class_weight="balanced_subsample", random_state=SEED, n_jobs=-1), Xs_leaky),
}
rows = []
for name, (mdl, feats) in cands.items():
    r = cross_validate(mdl, feats, y, cv=cv, scoring=["accuracy", "f1_macro"])
    rows.append(dict(model=name, accuracy=r["test_accuracy"].mean(), accuracy_sd=r["test_accuracy"].std(),
                     macro_f1=r["test_f1_macro"].mean(), macro_f1_sd=r["test_f1_macro"].std()))
sev = pd.DataFrame(rows); display(sev.round(3)); sev.to_csv(f"{RES_DIR}/severity_models.csv", index=False)

honest = sev[~sev.model.str.contains("LEAKY|baseline")].sort_values("macro_f1").iloc[-1]["model"]
SUMMARY["severity_best_honest"] = {"model": honest, "macro_f1": float(sev.set_index("model").loc[honest, "macro_f1"]),
                                   "accuracy": float(sev.set_index("model").loc[honest, "accuracy"])}
mdl, feats = cands[honest]
pred = cross_val_predict(mdl, feats, y, cv=cv)
ConfusionMatrixDisplay.from_predictions(y, pred, labels=["Low", "Medium", "High", "Extreme"], cmap="Blues", colorbar=False)
plt.title(f"{honest}: cross-validated confusion matrix"); savefig("04_severity_confusion"); plt.show()

rf = RandomForestClassifier(300, min_samples_leaf=3, class_weight="balanced_subsample", random_state=SEED, n_jobs=-1).fit(Xs, y)
imp = pd.Series(rf.feature_importances_, index=Xs.columns).sort_values(ascending=False).head(12)
imp[::-1].plot(kind="barh", title="Random forest feature importance (no victim counts)"); savefig("05_severity_importance"); plt.show()

## 6. Association rule mining

Items are grouped into *families* (duration, proven victims, era, region, theme). Rules whose antecedent and consequent share a family (e.g. "1–5 possible victims → 1–5 proven victims") are trivial and are filtered out.

In [ ]:
from mlxtend.frequent_patterns import apriori, association_rules

arm = pd.DataFrame(index=m.index)
arm["duration"] = pd.cut(m["duration"], [0, 1, 5, 10, 20, 500], labels=["1y", "2-5y", "6-10y", "11-20y", "20y+"])
arm["proven"] = pd.cut(m["proven"], [-1, 1, 5, 10, 20, 50, np.inf], labels=["1", "2-5", "6-10", "11-20", "21-50", "50+"])
arm["era"] = era.astype(str)
arm["region"] = m["region"]
A = pd.get_dummies(arm.astype(str), prefix_sep="=", dtype=bool)
for t in THEMES:
    A["theme=" + t] = m["theme_" + t].astype(bool)

items = apriori(A, min_support=0.03, use_colnames=True)
try:   # newer mlxtend versions require num_itemsets; older ones reject it
    rules = association_rules(items, num_itemsets=len(A), metric="lift", min_threshold=1.2)
except TypeError:
    rules = association_rules(items, metric="lift", min_threshold=1.2)
fam = lambda s: {i.split("=")[0] for i in s}
rules = rules[[len(fam(a) & fam(c)) == 0 for a, c in zip(rules["antecedents"], rules["consequents"])]]
rules = rules[rules["confidence"] >= 0.5].sort_values("lift", ascending=False)
show = rules.assign(antecedents=rules["antecedents"].apply(lambda s: " & ".join(sorted(s))),
                    consequents=rules["consequents"].apply(lambda s: " & ".join(sorted(s))))
print(f"{len(rules)} non-trivial rules (support>=3%, confidence>=50%, lift>=1.2)")
display(show[["antecedents", "consequents", "support", "confidence", "lift"]].head(15).round(3))
show[["antecedents", "consequents", "support", "confidence", "lift"]].to_csv(f"{RES_DIR}/association_rules.csv", index=False)

## 7. Time series (ARIMA) with honest validation

Fixes vs v1: the yearly series is built from **all** rows with a parsed start year, with **no imputation** (v1 filled unparsable years with 1981, which manufactured part of the "1980s peak"). The model is evaluated on a **held-out final window** against naive baselines, with (p,d,q) chosen by AIC. Important caveat: recent offenders may be uncaught or not yet documented (right-censoring), so the post-2000 decline is partly a data artefact.

In [ ]:
yr = df["start_year"].dropna().astype(int)
yr = yr[(yr >= 1900) & (yr <= 2022)]
counts = yr.value_counts().reindex(range(1900, 2023), fill_value=0).sort_index()
print("Top 5 years by number of offenders starting:"); print(counts.sort_values(ascending=False).head(5).to_string())

fig, ax = plt.subplots(figsize=(12, 4))
ax.plot(counts.index, counts.values); ax.axvspan(2005, 2022, color="grey", alpha=.15, label="likely right-censored")
ax.set_title("Documented offenders per start year"); ax.legend(); savefig("06_counts_per_year"); plt.show()

In [ ]:
from statsmodels.tsa.stattools import adfuller
from statsmodels.tsa.arima.model import ARIMA
from statsmodels.stats.diagnostic import acorr_ljungbox
import itertools

series = counts.loc[1900:2015].astype(float)                  # stop at 2015 to limit right-censoring
ts = pd.Series(series.values, index=pd.date_range("1900", periods=len(series), freq="YS"))
adf_p = adfuller(ts)[1]; d = 1 if adf_p > 0.05 else 0
print(f"ADF p-value = {adf_p:.4f} -> d = {d}")

H = 12
train, test = ts.iloc[:-H], ts.iloc[-H:]
aic = {}
for p, q in itertools.product(range(4), range(4)):
    try:
        aic[(p, d, q)] = ARIMA(train, order=(p, d, q)).fit().aic
    except Exception:
        pass
best_order = min(aic, key=aic.get)
print("Best order by AIC:", best_order, "| AIC =", round(aic[best_order], 1))

fit = ARIMA(train, order=best_order).fit()
fc = fit.get_forecast(H); mean_fc = fc.predicted_mean; ci = fc.conf_int()
mae = lambda a, b: float(np.mean(np.abs(np.asarray(a) - np.asarray(b))))
rmse = lambda a, b: float(np.sqrt(np.mean((np.asarray(a) - np.asarray(b)) ** 2)))
naive_last = np.repeat(train.iloc[-1], H); naive_mean5 = np.repeat(train.iloc[-5:].mean(), H)
res = pd.DataFrame({"MAE": [mae(test, mean_fc), mae(test, naive_last), mae(test, naive_mean5)],
                    "RMSE": [rmse(test, mean_fc), rmse(test, naive_last), rmse(test, naive_mean5)]},
                   index=[f"ARIMA{best_order}", "Naive (last value)", "Naive (mean of last 5)"])
display(res.round(2)); res.to_csv(f"{RES_DIR}/arima_holdout.csv")
print("Ljung-Box p (lag 10) on residuals:", round(float(acorr_ljungbox(fit.resid, lags=[10])["lb_pvalue"].iloc[0]), 3))

plt.figure(figsize=(12, 4.5))
plt.plot(ts, label="Actual"); plt.plot(mean_fc, "--", label="Forecast (held-out)")
plt.fill_between(ci.index, ci.iloc[:, 0], ci.iloc[:, 1], alpha=.2); plt.legend(); plt.title("ARIMA hold-out forecast")
savefig("07_arima_holdout"); plt.show()

pre70, post70 = counts.loc[:1969].mean(), counts.loc[1970:2015].mean()
print(f"Offenders starting per year: before 1970 = {pre70:.2f}, 1970-2015 = {post70:.2f}  (killers per year, NOT murders)")
SUMMARY["arima"] = {"order": list(best_order), "holdout_rmse": res["RMSE"].round(2).to_dict(), "pre1970_mean": float(pre70), "1970_2015_mean": float(post70)}

## 8. Save results

In [ ]:
m.drop(columns=["Notes"]).to_csv(f"{RES_DIR}/model_sample_with_clusters.csv", index=False)
with open(f"{RES_DIR}/summary.json", "w") as f:
    json.dump(SUMMARY, f, indent=2, default=str)
print(json.dumps(SUMMARY, indent=2, default=str))